# Construcción del Frontend para una Aplicación Full-Stack con LangGraph

## Guía Paso a Paso: Una Interfaz de Chat para el Agente de Atención al Cliente

En el notebook anterior (`065-LG1-top51-backend.ipynb`), creamos un **backend con FastAPI** que envuelve el agente de atención al cliente de LangGraph del notebook 028. El backend expone un endpoint `POST /chat` que recibe un mensaje y devuelve la respuesta del agente.

Ahora necesitamos la otra mitad: un **frontend** — una página web donde los clientes puedan escribir mensajes y ver las respuestas del agente. Esto es lo que el cliente realmente ve e interactúa.

Lo construiremos usando **Next.js** (React), el mismo framework utilizado en el frontend de la aplicación To Do (`051-todoapp-LC-frontend`).

**En este notebook, crearemos el frontend paso a paso.**

## ¿Qué es un Frontend? (Repaso Rápido)

El frontend es el **sitio web** que el cliente ve en su navegador. Tiene dos funciones:

1. **Mostrar** una ventana de chat donde el cliente puede escribir mensajes.
2. **Comunicarse** con el backend enviando peticiones HTTP y mostrando las respuestas.

```
┌──────────────────────────────────────────────────┐
│                    Navegador                     │
│                                                  │
│  ┌─────────────────────────────────────────────┐ │
│  │  BigCompany Inc. Atención al Cliente        │ │
│  │                                             │ │
│  │  ┌──────────────────────────────────┐       │ │
│  │  │Agente: ¡Hola! ¿En qué puedo      │       │ │
│  │  │ayudarle?                         │       │ │
│  │  └──────────────────────────────────┘       │ │
│  │       ┌───────────────────────────────────┐ │ │
│  │       │ Tú: ¿Cuál es el estado de ORD-1002│ │ │
│  │       └───────────────────────────────────┘ │ │
│  │  ┌──────────────────────────────────┐       │ │
│  │  │ Agente: Su pedido está enviado...│       │ │
│  │  └──────────────────────────────────┘       │ │
│  │                                             │ │
│  │  ┌──────────────────────────────┬────────┐  │ │
│  │  │ Escribe tu mensaje...        │ Enviar │  │ │
│  │  └──────────────────────────────┴────────┘  │ │
│  └─────────────────────────────────────────────┘ │
└──────────────────────────────────────────────────┘
```

El frontend **no** contiene ninguna lógica de IA. Simplemente envía el mensaje del cliente al backend, espera la respuesta y la muestra.

## ¿Cómo se Compara con el Frontend de la Aplicación To Do?

El frontend de la aplicación To Do (`051-todoapp-LC-frontend`) muestra una lista de tareas. Los usuarios pueden añadir, editar, eliminar y filtrar tareas. Cada acción envía una petición HTTP al backend.

Nuestro frontend de chat muestra una conversación. Los usuarios escriben mensajes y reciben respuestas. Cada mensaje envía **una petición HTTP** al backend (`POST /chat`).

| Característica | Frontend App To Do (051) | Nuestro Frontend de Chat |
|----------------|--------------------------|-------------------------|
| Framework | Next.js (React) | Next.js (React) |
| Componente principal | `todo-list.js` (lista de tareas) | `chat.js` (ventana de chat) |
| Acción del usuario | Añadir/editar/eliminar tareas | Enviar mensajes |
| Llamadas a la API | Múltiples (POST, GET, PUT, DELETE) | Una (POST /chat) |
| Hooks de React | `useState`, `useEffect`, `useCallback`, `useRef` | `useState`, `useRef`, `useEffect` |
| Estilos | CSS Modules | CSS Modules |

Mismas herramientas, mismos patrones, diferente propósito.

## Estructura del Proyecto

Estos son los archivos que crearemos:

```
frontend/
├── .env                      # URL de la API del backend
├── package.json              # Dependencias (next, react)
├── next.config.mjs           # Configuración de Next.js
├── jsconfig.json             # Alias de rutas en JavaScript
├── pages/
│   ├── _app.js               # Envoltorio de la app (carga CSS global)
│   ├── _document.js          # Estructura del documento HTML
│   └── index.js              # Página principal
├── components/
│   ├── layout.js             # Diseño de la página (título, centrado)
│   └── chat.js               # El componente de chat (¡la estrella del show!)
└── styles/
    ├── globals.css            # Estilos globales (body, fuentes)
    ├── layout.module.css      # Estilos del layout
    └── chat.module.css        # Estilos del componente de chat
```

Compara con el frontend de la aplicación To Do (`051-todoapp-LC-frontend/047-todoapp-frontend`):

```
047-todoapp-frontend/
├── .env                      # URL de la API del backend   ← mismo rol
├── package.json              # Dependencias                ← mismo rol
├── next.config.mjs           # Configuración de Next.js    ← mismo rol
├── jsconfig.json             # Alias de rutas              ← mismo rol
├── pages/
│   ├── _app.js               # Envoltorio de la app        ← mismo rol
│   ├── _document.js          # Documento HTML              ← mismo rol
│   └── index.js              # Página principal            ← mismo rol
├── components/
│   ├── layout.js             # Layout                      ← mismo rol
│   ├── todo-list.js          # Componente de la lista To Do ← reemplazado por chat.js
│   └── todo.js               # Elemento individual To Do   ← no necesario (no tenemos elementos)
└── styles/
    ├── globals.css            # Estilos globales            ← mismo rol
    ├── layout.module.css      # Estilos del layout          ← mismo rol
    ├── todo-list.module.css   # Estilos de la lista To Do   ← reemplazado por chat.module.css
    └── todo.module.css        # Estilos del elemento To Do  ← no necesario
```

La estructura es casi idéntica. La principal diferencia es que `todo-list.js` y `todo.js` se reemplazan por un único `chat.js`.

¡Construyámoslo archivo por archivo!

## Paso 1: Crear las Carpetas del Proyecto

Necesitamos tres carpetas: `pages/`, `components/` y `styles/`.

In [ ]:
import os

# Crear las carpetas del frontend
os.makedirs("frontend/pages", exist_ok=True)
os.makedirs("frontend/components", exist_ok=True)
os.makedirs("frontend/styles", exist_ok=True)
os.makedirs("frontend/public", exist_ok=True)

print("¡Carpetas creadas!")
print("  frontend/")
print("  frontend/pages/")
print("  frontend/components/")
print("  frontend/styles/")
print("  frontend/public/")

## Paso 2: Crear el Archivo `.env` (URL de la API del Backend)

El frontend necesita saber **dónde está ejecutándose el backend** para poder enviarle mensajes.

En Next.js, las variables de entorno que comienzan con `NEXT_PUBLIC_` están disponibles en el navegador. Usamos `NEXT_PUBLIC_API_URL` para almacenar la URL del backend.

Esto es exactamente igual que el archivo `.env` de la aplicación To Do.

In [ ]:
%%writefile frontend/.env
NEXT_PUBLIC_API_URL=http://localhost:8000

### Comparación con el `.env` de la aplicación To Do

Aplicación To Do: `NEXT_PUBLIC_API_URL=http://localhost:8000`

Nuestra aplicación de chat: `NEXT_PUBLIC_API_URL=http://localhost:8000`

¡Idénticos! Ambos frontends se comunican con un backend en el puerto 8000.

## Paso 3: Crear `package.json` (Dependencias)

Este archivo lista los paquetes de JavaScript que nuestro frontend necesita y define los comandos para ejecutarlo.

La aplicación To Do necesitaba `lodash` (para hacer debounce en los cambios de entrada). Nuestra aplicación de chat no necesita ninguna librería adicional — solo **Next.js** y **React**.

| Paquete | Qué Hace |
|---------|----------|
| `next` | El framework de React (gestiona el enrutamiento, renderizado del lado del servidor, etc.) |
| `react` | La librería de interfaz de usuario (componentes, hooks, gestión de estado) |
| `react-dom` | Renderiza los componentes de React en el navegador |

In [ ]:
%%writefile frontend/package.json
{
  "name": "bigcompany-chat-frontend",
  "version": "0.1.0",
  "private": true,
  "scripts": {
    "dev": "next dev",
    "build": "next build",
    "start": "next start"
  },
  "dependencies": {
    "next": "16.1.4",
    "react": "19.2.3",
    "react-dom": "19.2.3"
  }
}

### Explicación de los scripts

| Comando | Qué Hace |
|---------|----------|
| `npm run dev` | Inicia el servidor de desarrollo (con recarga automática) |
| `npm run build` | Construye la aplicación para producción |
| `npm start` | Ejecuta la versión de producción |

Durante el desarrollo, usaremos `npm run dev`. Se inicia en **http://localhost:3000**.

## Paso 4: Crear los Archivos de Configuración

Estos son pequeños archivos de configuración que Next.js necesita. Son los mismos que los archivos de configuración de la aplicación To Do.

In [ ]:
%%writefile frontend/next.config.mjs
/** @type {import('next').NextConfig} */
const nextConfig = {
  reactStrictMode: true,
};

export default nextConfig;

In [ ]:
%%writefile frontend/jsconfig.json
{
  "compilerOptions": {
    "paths": {
      "@/*": ["./*"]
    }
  }
}

- **`next.config.mjs`**: Activa el Modo Estricto de React, que ayuda a detectar errores comunes durante el desarrollo.
- **`jsconfig.json`**: Configura el alias de ruta `@/`, para que podamos escribir `import styles from '@/styles/globals.css'` en lugar de `import styles from '../../styles/globals.css'`.

Estos son idénticos a los archivos de configuración de la aplicación To Do.

## Paso 5: Crear la Estructura de Páginas

En Next.js, la carpeta `pages/` define las rutas del sitio web:
- `pages/index.js` → la página principal (`/`)
- `pages/_app.js` → envuelve cada página (carga estilos globales)
- `pages/_document.js` → la estructura del documento HTML

Estos archivos son el **mismo código base** en cada aplicación Next.js. La aplicación To Do tiene exactamente los mismos archivos.

In [ ]:
%%writefile frontend/pages/_app.js
import "@/styles/globals.css";

export default function App({ Component, pageProps }) {
  return <Component {...pageProps} />;
}

In [ ]:
%%writefile frontend/pages/_document.js
import { Html, Head, Main, NextScript } from "next/document";

export default function Document() {
  return (
    <Html lang="en">
      <Head />
      <body>
        <Main />
        <NextScript />
      </body>
    </Html>
  );
}

- **`_app.js`**: Carga el archivo CSS global para que los estilos estén disponibles en todas las páginas. Es idéntico al de la aplicación To Do.
- **`_document.js`**: Define la estructura del documento HTML (`<html>`, `<head>`, `<body>`). Es idéntico al de la aplicación To Do.

## Paso 6: Crear los Estilos

Necesitamos tres archivos CSS:

1. **`globals.css`** — Estilos base para toda la página (fuente, fondo).
2. **`layout.module.css`** — Estilos para el envoltorio del layout (centrado, título).
3. **`chat.module.css`** — Estilos para el componente de chat (mensajes, entrada, burbujas).

Usamos **CSS Modules** (archivos que terminan en `.module.css`). Es el mismo enfoque que usa la aplicación To Do. Los CSS Modules mantienen los estilos limitados a su componente, para que no afecten accidentalmente a otras partes de la página.

In [ ]:
%%writefile frontend/styles/globals.css
* {
  box-sizing: border-box;
  margin: 0;
  padding: 0;
}

body {
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
  background-color: #fafafa;
  display: flex;
  justify-content: center;
  padding-top: 30px;
}

In [ ]:
%%writefile frontend/styles/layout.module.css
.layout {
  width: 500px;
}

.title {
  text-align: center;
  font-size: 24px;
  margin-bottom: 15px;
  color: #333;
}

Ahora el archivo CSS más importante — los estilos para el componente de chat. Esto crea la apariencia de la ventana de chat: burbujas de mensajes, el área de entrada y un indicador de carga.

Compara con la aplicación To Do:
- La aplicación To Do tenía `todo-list.module.css` (contenedor, entrada, botones de filtro) y `todo.module.css` (fila del todo, checkbox, botón eliminar).
- Nuestra aplicación tiene `chat.module.css` (contenedor, área de mensajes, formulario de entrada, burbujas de mensajes).

In [ ]:
%%writefile frontend/styles/chat.module.css
/* ============================================================
   Estilos del Componente de Chat
   ============================================================ */

/* El contenedor principal del chat — una caja alta con mensajes arriba y entrada abajo */
.container {
  width: 500px;
  height: 600px;
  border: 1px solid #ddd;
  border-radius: 8px;
  background-color: white;
  display: flex;
  flex-direction: column;
}

/* El área desplazable donde aparecen los mensajes */
.messages {
  flex: 1;
  overflow-y: auto;
  padding: 20px;
}

/* Se muestra cuando aún no hay mensajes */
.emptyState {
  text-align: center;
  color: #999;
  margin-top: 200px;
  padding: 0 40px;
  line-height: 1.6;
}

/* Cada burbuja de mensaje */
.message {
  margin-bottom: 15px;
  padding: 10px 15px;
  border-radius: 8px;
  max-width: 85%;
}

/* Mensajes del cliente: fondo azul, alineados a la derecha */
.user {
  background-color: #e3f2fd;
  margin-left: auto;
}

/* Mensajes del agente: fondo gris, alineados a la izquierda */
.agent {
  background-color: #f5f5f5;
  margin-right: auto;
}

/* La etiqueta encima de cada mensaje ("Tú" o "Agente") */
.sender {
  font-size: 11px;
  font-weight: bold;
  margin-bottom: 4px;
  color: #888;
}

/* El texto del mensaje */
.content {
  line-height: 1.5;
  white-space: pre-wrap;
  font-size: 14px;
}

/* El formulario de entrada en la parte inferior */
.inputForm {
  display: flex;
  padding: 15px;
  border-top: 1px solid #ddd;
  gap: 10px;
}

/* El campo de texto */
.input {
  flex: 1;
  padding: 10px;
  border: 1px solid #ddd;
  border-radius: 4px;
  font-size: 14px;
  outline: none;
}

.input:focus {
  border-color: #4a90d9;
}

/* El botón de enviar */
.sendBtn {
  padding: 10px 20px;
  background-color: #4a90d9;
  color: white;
  border: none;
  border-radius: 4px;
  cursor: pointer;
  font-size: 14px;
}

.sendBtn:hover {
  background-color: #357abd;
}

/* Estado desactivado mientras se espera una respuesta */
.sendBtn:disabled {
  background-color: #ccc;
  cursor: not-allowed;
}

## Paso 7: Crear `components/layout.js` (El Layout)

El componente de layout envuelve el contenido de la página y añade un título. Es el mismo patrón que el `layout.js` de la aplicación To Do.

In [ ]:
%%writefile frontend/components/layout.js
import styles from '../styles/layout.module.css'

export default function Layout(props) {
  return (
    <div className={styles.layout}>
      <h1 className={styles.title}>BigCompany Inc. Support</h1>
      {props.children}
    </div>
  )
}

### Comparación con el layout de la aplicación To Do

Aplicación To Do:
```jsx
export default function Layout(props) {
  return (
    <div className={styles.layout}>
      <h1 className={styles.title}>To Do</h1>
      {props.children}
    </div>
  )
}
```

Nuestra aplicación de chat:
```jsx
export default function Layout(props) {
  return (
    <div className={styles.layout}>
      <h1 className={styles.title}>BigCompany Inc. Support</h1>
      {props.children}
    </div>
  )
}
```

La única diferencia es el texto del título. Todo lo demás es igual.

## Paso 8: Crear `components/chat.js` (¡La Estrella del Show!)

Este es el **componente principal** de nuestro frontend — la ventana de chat. Reemplaza al `todo-list.js` de la aplicación To Do.

Esto es lo que hace este componente:

1. **Mantiene una lista de mensajes** en el estado (usando `useState`).
2. **Muestra los mensajes** en un área desplazable (mensajes del cliente a la derecha, mensajes del agente a la izquierda).
3. **Tiene un campo de texto** y un botón Enviar en la parte inferior.
4. **Cuando el usuario envía un mensaje**: lo añade a la lista, lo envía al backend vía `POST /chat`, espera la respuesta y añade la respuesta del agente a la lista.
5. **Muestra "Pensando..."** mientras espera la respuesta del agente.
6. **Auto-desplazamiento** hacia abajo cuando llegan nuevos mensajes.
7. **Genera un ID de hilo único** para cada carga de la página, para que cada visita inicie una conversación nueva.

### Los conceptos clave de React utilizados

| Concepto | Qué Hace | ¿También se usa en la App To Do? |
|----------|----------|----------------------------------|
| `useState` | Almacena la lista de mensajes, el texto de entrada y el estado de carga | Sí (almacena todos, entrada, filtro) |
| `useRef` | Crea una referencia al final del área de mensajes (para auto-desplazamiento) | Sí (didFetchRef) |
| `useEffect` | Auto-desplaza hacia abajo cuando los mensajes cambian | Sí (obtiene los todos al montar) |
| `fetch()` | Envía peticiones HTTP al backend | Sí (todas las operaciones CRUD) |
| `async/await` | Gestiona la llamada asíncrona a la API | Sí (fetchTodos, addToDo, etc.) |

In [ ]:
%%writefile frontend/components/chat.js
// ============================================================
// chat.js — El Componente de Chat
//
// Este es el componente principal de nuestro frontend.
// Muestra una ventana de chat donde los clientes pueden hablar
// con el agente de atención al cliente de BigCompany Inc.
// ============================================================

import { useState, useRef, useEffect } from 'react'
import styles from '../styles/chat.module.css'

export default function Chat() {
  // ---- ESTADO ----
  // messages: la lista de todos los mensajes en la conversación
  //   Cada mensaje es un objeto: { role: 'user' o 'agent', content: 'el texto' }
  const [messages, setMessages] = useState([])

  // input: lo que el usuario está escribiendo actualmente en el campo de texto
  const [input, setInput] = useState('')

  // isLoading: true mientras se espera la respuesta del agente
  const [isLoading, setIsLoading] = useState(false)

  // threadId: un ID único para esta conversación
  //   Se genera una vez cuando el componente se carga.
  //   Mismo threadId = misma conversación (el agente recuerda mensajes anteriores).
  const [threadId] = useState(() => 'thread-' + Date.now())

  // messagesEndRef: una referencia al final del área de mensajes
  //   Se usa para auto-desplazar al último mensaje.
  const messagesEndRef = useRef(null)

  // ---- AUTO-DESPLAZAMIENTO ----
  // Cada vez que la lista de mensajes cambia, desplazar hacia abajo.
  useEffect(() => {
    messagesEndRef.current?.scrollIntoView({ behavior: 'smooth' })
  }, [messages, isLoading])

  // ---- ENVIAR MENSAJE ----
  // Esta función se ejecuta cuando el usuario hace clic en Enviar o presiona Enter.
  async function sendMessage(e) {
    e.preventDefault()

    // No enviar mensajes vacíos ni enviar mientras ya se está cargando
    if (!input.trim() || isLoading) return

    const userMessage = input.trim()
    setInput('')  // Limpiar el campo de entrada

    // Paso 1: Añadir el mensaje del usuario a la lista inmediatamente
    //   (para que el usuario vea su mensaje de inmediato)
    setMessages(prev => [...prev, { role: 'user', content: userMessage }])

    // Paso 2: Enviar el mensaje al backend
    setIsLoading(true)

    try {
      const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/chat', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({
          message: userMessage,
          thread_id: threadId,
        }),
      })

      if (res.ok) {
        // Paso 3: Añadir la respuesta del agente a la lista
        const data = await res.json()
        setMessages(prev => [...prev, { role: 'agent', content: data.response }])
      } else {
        // Algo salió mal en el servidor
        setMessages(prev => [...prev, {
          role: 'agent',
          content: 'Lo siento, algo salió mal. Por favor, inténtelo de nuevo.'
        }])
      }
    } catch (error) {
      // No se pudo conectar al backend
      setMessages(prev => [...prev, {
        role: 'agent',
        content: 'No se pudo conectar al servidor. ¿Está el backend ejecutándose en ' +
          process.env.NEXT_PUBLIC_API_URL + '?'
      }])
    }

    setIsLoading(false)
  }

  // ---- RENDERIZADO ----
  return (
    <div className={styles.container}>

      {/* Área de mensajes (desplazable) */}
      <div className={styles.messages}>

        {/* Estado vacío: se muestra cuando aún no hay mensajes */}
        {messages.length === 0 && (
          <div className={styles.emptyState}>
            ¡Bienvenido al soporte de BigCompany Inc.!<br />
            Envíe un mensaje para empezar a chatear con nuestro agente de IA.
          </div>
        )}

        {/* Renderizar cada mensaje */}
        {messages.map((msg, index) => (
          <div
            key={index}
            className={`${styles.message} ${msg.role === 'user' ? styles.user : styles.agent}`}
          >
            <div className={styles.sender}>
              {msg.role === 'user' ? 'Tú' : 'Agente'}
            </div>
            <div className={styles.content}>{msg.content}</div>
          </div>
        ))}

        {/* Indicador de carga: se muestra mientras se espera al agente */}
        {isLoading && (
          <div className={`${styles.message} ${styles.agent}`}>
            <div className={styles.sender}>Agente</div>
            <div className={styles.content}>Pensando...</div>
          </div>
        )}

        {/* Elemento invisible en la parte inferior — usado para auto-desplazamiento */}
        <div ref={messagesEndRef} />
      </div>

      {/* Formulario de entrada en la parte inferior */}
      <form className={styles.inputForm} onSubmit={sendMessage}>
        <input
          className={styles.input}
          type="text"
          placeholder="Escribe tu mensaje..."
          value={input}
          onChange={(e) => setInput(e.target.value)}
          disabled={isLoading}
        />
        <button
          className={styles.sendBtn}
          type="submit"
          disabled={isLoading || !input.trim()}
        >
          Enviar
        </button>
      </form>
    </div>
  )
}

### Entendiendo `chat.js` — Las Partes Clave

Desglosemos las partes más importantes de este componente:

#### 1. Gestión del Estado
```jsx
const [messages, setMessages] = useState([])    // Lista de mensajes
const [input, setInput] = useState('')           // Texto de entrada actual
const [isLoading, setIsLoading] = useState(false) // ¿Está pensando el agente?
const [threadId] = useState(() => 'thread-' + Date.now())  // ID único de conversación
```

La aplicación To Do tenía un estado similar: `const [todos, setTodos] = useState(null)` y `const [mainInput, setMainInput] = useState('')`.

#### 2. Enviar un Mensaje al Backend
```jsx
const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/chat', {
  method: 'POST',
  headers: { 'Content-Type': 'application/json' },
  body: JSON.stringify({ message: userMessage, thread_id: threadId }),
})
const data = await res.json()  // { response: "...", thread_id: "..." }
```

Este es el mismo patrón que la función `addToDo()` de la aplicación To Do:
```jsx
const res = await fetch(process.env.NEXT_PUBLIC_API_URL + '/todos/', {
  method: 'POST',
  body: JSON.stringify({ name: name, completed: false }),
  headers: { 'Content-Type': 'application/json' }
})
```

Mismo patrón con `fetch()`, diferente endpoint y datos.

#### 3. Auto-Desplazamiento
```jsx
const messagesEndRef = useRef(null)

useEffect(() => {
  messagesEndRef.current?.scrollIntoView({ behavior: 'smooth' })
}, [messages, isLoading])
```

Colocamos un `<div ref={messagesEndRef} />` invisible en la parte inferior de los mensajes. Cada vez que la lista de mensajes cambia, `useEffect` desplaza ese div invisible a la vista. Esto asegura que el último mensaje siempre sea visible.

## Paso 9: Crear `pages/index.js` (La Página Principal)

Esta es la página principal de nuestro sitio web. Ensambla el layout y el componente de chat juntos.

Sigue exactamente el mismo patrón que el `index.js` de la aplicación To Do.

In [ ]:
%%writefile frontend/pages/index.js
import Head from 'next/head'
import Layout from '../components/layout'
import Chat from '../components/chat'

export default function Home() {
  return (
    <div>
      <Head>
        <title>BigCompany Inc. Customer Support</title>
        <meta name="description" content="BigCompany Inc. AI Customer Support" />
        <link rel="icon" href="/favicon.ico" />
      </Head>
      <Layout>
        <Chat />
      </Layout>
    </div>
  )
}

### Comparación con el `index.js` de la aplicación To Do

Aplicación To Do:
```jsx
import Layout from '../components/layout';
import ToDoList from '../components/todo-list';

export default function Home() {
  return (
    <div>
      <Head>
        <title>Full Stack Book To Do</title>
      </Head>
      <Layout>
        <ToDoList />
      </Layout>
    </div>
  )
}
```

Nuestra aplicación de chat:
```jsx
import Layout from '../components/layout'
import Chat from '../components/chat'

export default function Home() {
  return (
    <div>
      <Head>
        <title>BigCompany Inc. Customer Support</title>
      </Head>
      <Layout>
        <Chat />
      </Layout>
    </div>
  )
}
```

Las únicas diferencias: el título y `<ToDoList />` se reemplaza por `<Chat />`.

## ¡Todos los Archivos Creados! Revisemos

Verifiquemos que todos nuestros archivos del frontend se crearon correctamente.

In [ ]:
import os

print("Estructura del proyecto frontend:")
print("=" * 40)
for root, dirs, files in os.walk("frontend"):
    # Omitir carpetas ocultas y node_modules
    dirs[:] = [d for d in dirs if not d.startswith(".") and d != "node_modules" and d != ".next"]
    level = root.replace("frontend", "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    print(f"{indent}{folder_name}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        if not file.startswith(".") or file in [".env"]:
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath)
            print(f"{sub_indent}{file} ({size} bytes)")

## Paso 10: Instalar Dependencias y Ejecutar el Frontend

### Paso 10a: Instalar los paquetes de JavaScript

Abre un terminal y ejecuta:

```bash
cd frontend
npm install
```

Esto lee `package.json` y descarga todos los paquetes (Next.js, React, etc.) en una carpeta `node_modules/`. Puede tardar un minuto.

### Paso 10b: Iniciar el backend (si no está ya en ejecución)

El frontend necesita que el backend esté ejecutándose para poder enviar mensajes. En un **terminal separado**:

```bash
cd backend
uvicorn main:app --reload
```

El backend se inicia en **http://localhost:8000**.

### Paso 10c: Iniciar el frontend

En otro terminal:

```bash
cd frontend
npm run dev
```

El frontend se inicia en **http://localhost:3000**.

### Paso 10d: ¡Abrir el chat!

Abre tu navegador y ve a:

**http://localhost:3000**

Deberías ver la ventana de chat. Prueba estos mensajes:

1. `¡Hola! Necesito ayuda con mi pedido.`
2. `¿Puede comprobar el estado del pedido ORD-1002?`
3. `¿Cuándo llegará?` (¡prueba la memoria de la conversación!)
4. `¿Cuál es su política de devoluciones?`

## Cómo Funciona Todo Junto

Este es el flujo completo cuando un cliente escribe un mensaje:

```
1. El cliente escribe "Consultar pedido ORD-1002" y hace clic en Enviar
        │
        ▼
2. chat.js añade el mensaje a la pantalla inmediatamente
   y muestra "Pensando..."
        │
        ▼
3. chat.js envía POST http://localhost:8000/chat
   con {message: "Consultar pedido ORD-1002", thread_id: "thread-1707..."}
        │
        ▼
4. El backend FastAPI (main.py) recibe la petición
   └── La enruta a routers/chat.py
        │
        ▼
5. routers/chat.py llama a graph.invoke() desde agent.py
        │
        ▼
6. El agente LangGraph (agent.py):
   ├── Nodo Asistente: el LLM lee el mensaje
   ├── El LLM decide llamar a la herramienta check_order_status
   ├── Nodo Herramientas: obtiene los datos del pedido de la "base de datos"
   ├── Nodo Asistente: el LLM lee los resultados de la herramienta
   └── El LLM genera la respuesta final
        │
        ▼
7. El backend devuelve {response: "Su pedido está enviado...", thread_id: "thread-1707..."}
        │
        ▼
8. chat.js recibe la respuesta
   ├── Elimina "Pensando..."
   ├── Añade el mensaje del agente a la pantalla
   └── Auto-desplaza hacia abajo
        │
        ▼
9. ¡El cliente ve la respuesta del agente!
```

## Resumen: Lo Que Hemos Construido

Creamos un **frontend con Next.js** — una interfaz de chat para el agente de atención al cliente de BigCompany Inc.

| Archivo | Función |
|---------|--------|
| `.env` | Almacena la URL del backend (`http://localhost:8000`) |
| `package.json` | Lista las dependencias (next, react, react-dom) |
| `next.config.mjs` | Configuración de Next.js |
| `jsconfig.json` | Alias de rutas en JavaScript |
| `pages/_app.js` | Carga los estilos globales |
| `pages/_document.js` | Estructura del documento HTML |
| `pages/index.js` | Página principal — ensambla Layout + Chat |
| `components/layout.js` | Layout con título |
| `components/chat.js` | **El componente de chat** — mensajes, entrada, llamadas a la API |
| `styles/globals.css` | Estilos base de la página |
| `styles/layout.module.css` | Estilos del layout |
| `styles/chat.module.css` | Estilos del componente de chat |

### La Arquitectura Full-Stack

```
┌─────────────────────┐       HTTP        ┌─────────────────────┐       invoke      ┌──────────────────┐
│    Frontend         │  ─────────────►   │    Backend          │  ─────────────►   │  Agente LangGraph│
│    (Next.js)        │  POST /chat       │    (FastAPI)        │                   │  (agent.py)      │
│    localhost:3000   │  ◄─────────────   │    localhost:8000   │  ◄─────────────   │                  │
│                     │  Respuesta JSON   │                     │  respuesta agente │  Herramientas:   │
│  components/chat.js │                   │  routers/chat.py    │                   │  - check_order   │
│                     │                   │  agent.py           │                   │  - return_policy │
│                     │                   │  schemas.py         │                   │  - process_return│
└─────────────────────┘                   └─────────────────────┘                   └──────────────────┘
```

## Conclusiones Clave

1. **El frontend es solo una interfaz de chat.** Envía mensajes al backend y muestra las respuestas. Aquí no hay ninguna lógica de IA.

2. **La llamada `fetch()` es el puente.** La única petición `POST /chat` conecta el frontend con todo el agente LangGraph.

3. **Los mismos patrones que la aplicación To Do.** Usamos `useState` para el estado, `fetch()` para las llamadas a la API, CSS Modules para los estilos y `pages/index.js` para la página principal. Si entiendes la aplicación To Do, entiendes esto.

4. **Los IDs de hilo mantienen las conversaciones separadas.** Cada carga de página genera un nuevo `thread_id`. Refrescar la página inicia una conversación nueva.

5. **La gestión de errores es importante.** Gestionamos dos casos de error: errores del servidor (respuesta no-200) y errores de conexión (backend no ejecutándose). Esto da al usuario retroalimentación útil en lugar de una pantalla en blanco.

## Referencia Rápida: Ejecutar la Aplicación Full-Stack

Necesitas **dos terminales** ejecutándose al mismo tiempo:

**Terminal 1 — Backend:**
```bash
cd backend
uvicorn main:app --reload
# Se ejecuta en http://localhost:8000
```

**Terminal 2 — Frontend:**
```bash
cd frontend
npm run dev
# Se ejecuta en http://localhost:3000
```

Luego abre **http://localhost:3000** en tu navegador y ¡empieza a chatear!